## 1. Imports and Hardware Setup

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

Using device: cpu


In [5]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = './data'
    SAVE_PATH = '/content/drive/MyDrive/DL'
    print("✅ Contected to Google Drive")
except(ValueError, ImportError, Exception) as e:
    print(f"❌ Error: {e}")
    print("❌ Failed to mount to Google Drive --> ✅ Use Local Path")
    DATA_PATH = "./data"
    SAVE_PATH = DATA_PATH

Mounted at /content/drive
✅ Contected to Google Drive


## 2. Hyperparameters

In [ ]:
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 10

## 3. Dataset Preparation

In [8]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root=DATA_PATH, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=DATA_PATH, train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 63.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.70MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.8MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.80MB/s]


## 4. LeNet-5 Model Architecture

In [14]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5), # C1
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2), # S2

            nn.Conv2d(6, 16, kernel_size=5), # C3
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2), # S4

            nn.Conv2d(16, 120, kernel_size=5), # C5
            nn.Tanh()
        )

        self.classifier = nn.Sequential(
            nn.Linear(120, 84), # F6
            nn.Tanh(),
            nn.Linear(84, num_classes) # Output
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        out = self.classifier(x)

        return out

model = LeNet5().to(device)
print("✅ Model created")

✅ Model created


## 5. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
# Use SGD like in the Paper
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [ ]:
def train(loader, model, criterion, optimizer):
    model.train()
    train_loss, correct = 0, 0
    loop = tqdm(loader, desc="Training", leave=False)

    for x, y in loop:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()

    return train_loss / len(loader.dataset), correct / len(loader.dataset)


## 6. Results Visualization

In [7]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['test_loss'], label='Test Loss')
plt.title('Loss History')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['test_acc'], label='Test Acc')
plt.title('Accuracy History')
plt.legend()
plt.show()

## 7. Model Inference Demo

In [8]:
def visualize_inference(model, loader, device, num_images=10):
    model.eval()
    images, labels = next(iter(loader))
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    _, preds = torch.max(outputs, 1)
    
    plt.figure(figsize=(15, 3))
    for i in range(num_images):
        plt.subplot(1, num_images, i+1)
        plt.imshow(images[i].cpu().squeeze(), cmap='gray')
        plt.title(f"P: {preds[i].item()}, L: {labels[i].item()}")
        plt.axis('off')
    plt.show()

visualize_inference(model, test_loader, DEVICE)